# blueprint-vlm on Kaggle (2x T4)

Settings -> **Accelerator: GPU T4 x2**, **Internet: On**.

T4 is Turing and has no bfloat16, so this uses fp16 throughout. The two cards
are used as two DDP ranks with a full model copy each, not as one 32GB pool --
they are separate 16GB cards and a 3B model in 4-bit fits on one comfortably.

In [ ]:
# Preflight. Run this before anything else -- it is cheap and it catches the
# two failures that waste a whole session: no accelerator, and no internet.
import shutil, torch, socket

print('nvidia-smi :', shutil.which('nvidia-smi') or 'NOT FOUND (session is CPU-only)')
print('cuda       :', torch.cuda.is_available(), '| devices:', torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        d = torch.cuda.get_device_properties(i)
        print(f'  [{i}] {d.name}  {d.total_memory/1e9:.0f} GB  sm_{d.major}{d.minor}')
    print('bf16       :', torch.cuda.is_bf16_supported(), '(False on T4 -> fp16 path, expected)')
else:
    raise SystemExit(
        'No GPU attached.\n'
        '  Sidebar -> Session options -> Accelerator -> "GPU T4 x2", then re-run from the top.\n'
        '  Changing the accelerator restarts the session.\n'
        '  Greyed out? Kaggle needs a verified phone number before it grants GPUs.\n'
        '  Still nothing? Check the 30 hr/week GPU quota (resets Saturday).'
    )

try:
    socket.create_connection(('huggingface.co', 443), timeout=5)
    print('internet   : on')
except OSError:
    raise SystemExit('No internet -- sidebar -> Session options -> Internet: On. '
                     'Needed to pull the 7GB model from Hugging Face.')


In [ ]:
import os
os.environ['HF_HOME'] = '/kaggle/working/hf'     # keep the 7GB download on the big disk
os.environ['PYTHONPATH'] = 'src'
!git clone -q https://github.com/baohq1595/blueprint-vlm.git /kaggle/working/blueprint-vlm
%cd /kaggle/working/blueprint-vlm
!pip install -q -U transformers peft accelerate bitsandbytes qwen-vl-utils

In [ ]:
!python scripts/build_dataset.py --out data/synth

## 1. Zero-shot baseline
Single GPU is fine for inference. ~20 min for the 1600-record test split.

In [ ]:
!PYTHONPATH=src python -m blueprint_vlm.infer \
    --config configs/qwen25vl_3b_lora_t4.yaml \
    --split data/synth/test --out results/zeroshot_test.json --batch-size 2
!PYTHONPATH=src python scripts/score.py --split data/synth/test \
    --predictions results/zeroshot_test.json \
    --out results/zeroshot_test_scored.json --title 'zero-shot / test'

## 2. Fine-tune across both cards
4000 records x 2 epochs. Budget 3-5 hours; the session limit is 12.

In [ ]:
!PYTHONPATH=src accelerate launch --multi_gpu --num_processes 2 \
    --mixed_precision fp16 --module blueprint_vlm.train \
    --config configs/qwen25vl_3b_lora_t4.yaml

## 3. Score both held-out sets
`test_hard` is the shifted distribution. Report it even when it is bad.

In [ ]:
for split in ['test', 'test_hard']:
    !PYTHONPATH=src python -m blueprint_vlm.infer \
        --config configs/qwen25vl_3b_lora_t4.yaml --split data/synth/{split} \
        --adapter outputs/qwen25vl-3b-lora-t4 \
        --out results/finetuned_{split}.json --batch-size 2
    !PYTHONPATH=src python scripts/score.py --split data/synth/{split} \
        --predictions results/finetuned_{split}.json \
        --out results/finetuned_{split}_scored.json --title 'fine-tuned / {split}'

## 4. If you are short on time
Run the baseline and the fine-tune on `test` only. `test_hard` and the
resolution ablation can go in a second session -- the adapter is saved to
`/kaggle/working` and can be added as a dataset to the next notebook.